In [3]:
#PCFG
import nltk
from nltk import PCFG
from nltk.parse import InsideChartParser
grammar = PCFG.fromstring("""
S -> NP VP [1.0]
NP -> Det Noun [0.5]
NP -> NP PP [0.3]
NP -> 'Alice' [0.2]
VP -> Verb NP [0.6]
VP -> VP PP [0.4]
PP -> Pre NP [1.0]
Det -> 'the' [1.0]
Noun -> 'boy' [0.5] | 'telescope' [0.5]
Verb -> 'saw' [1.0]
Pre -> 'with' [1.0]
""")

sentence = "the boy saw Alice with the telescope".split()
parser = InsideChartParser(grammar)
trees = []
for t in parser.parse(sentence):
    trees.append(t)
print("Total Parse Trees:", len(trees))
i = 0
for t in trees:
    print("\nTree", i + 1)
    print("Probability:", t.prob())
    t.pretty_print()
    i = i + 1
inside_prob = 0
for t in trees:
    inside_prob = inside_prob + t.prob()
print("\nInside Probability:", inside_prob)

Total Parse Trees: 2

Tree 1
Probability: 0.003
                        S                         
      __________________|____                      
     |                       VP                   
     |              _________|____                 
     |             |              PP              
     |             |          ____|___             
     NP            VP        |        NP          
  ___|___      ____|____     |     ___|______      
Det     Noun Verb       NP  Pre  Det        Noun  
 |       |    |         |    |    |          |     
the     boy  saw      Alice with the     telescope


Tree 2
Probability: 0.00225
                    S                             
      ______________|____                          
     |                   VP                       
     |         __________|____                     
     |        |               NP                  
     |        |      _________|___                 
     |        |     |             PP           

In [4]:
#CYK
import pandas as pd
import numpy as np
from collections import defaultdict
from nltk.tokenize import word_tokenize
from nltk.tree import Tree
rules = {
    "S": [("NP", "VP", 1.0)],
    "NP": [
        ("Det", "Noun", 0.5),
        ("NP", "PP", 0.5)
    ],
    "VP": [
        ("Verb", "NP", 0.6),
        ("VP", "PP", 0.4)
    ],
    "PP": [("Pre", "NP", 1.0)],
    "Det": [
        ("the", 0.5),
        ("a", 0.5)
    ],
    "Noun": [
        ("burglar", 0.4),
        ("man", 0.3),
        ("knife", 0.3)
    ],
    "Verb": [("threatened", 1.0)],
    "Pre": [("with", 1.0)]
}
sentence = "the burglar threatened the man with the knife"
sentence = word_tokenize(sentence)
n1 = len(sentence)
table = defaultdict(list)
for i, j in enumerate(sentence):
    for k, l in rules.items():
        for n in l:
            if len(n) == 2:
                o, p = n
                if o == j:
                    tree = Tree(k, [j])
                    table[(i, i, k)].append((tree, p))
for l in range(2, n1 + 1):
    for start in range(n1 - l + 1):
        end = start + l - 1
        for split in range(start, end):
            for i, j in rules.items():
                for r in j:
                    if len(r) == 3:
                        left, right, prob = r
                        left_p = table[(start, split, left)]
                        right_p = table[(split + 1, end, right)]
                        for a, b in left_p:
                            for c, d in right_p:
                                p = prob * b * d
                                tree = Tree(i, [a, c])
                                table[(start, end, i)].append((tree, p))

parses = table[(0, n1 - 1, "S")]
p = 0
for tree, prob in parses:
    p += prob
    tree.pretty_print()
    print("Probability:", prob)
print("\nInside Probability:", p)

                            S                             
      ______________________|_______                       
     |                              VP                    
     |               _______________|____                  
     |              |                    NP               
     |              |            ________|____             
     |              |           |             PP          
     |              |           |         ____|___         
     NP             |           NP       |        NP      
  ___|_____         |        ___|___     |     ___|____    
Det       Noun     Verb    Det     Noun Pre  Det      Noun
 |         |        |       |       |    |    |        |   
the     burglar threatened the     man  with the     knife

Probability: 0.00016875
                            S                             
      ______________________|_______                       
     |                              VP                    
     |                  